In [ ]:
import gzip
import json
import re
import pandas as pd
from lxml import etree
from pathlib import Path
from difflib import SequenceMatcher

def normalize_for_matching(text):
    """Normalize text for comparison"""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_vila_sections(json_content):
    """Extract VILA headings (in ALL-CAPS) with their corresponding text sections"""
    pattern = r'(^\s*(?:\d+\.?\d*\s+)?[A-Z][A-Z\s]+)(.*?)(?=\n\s*(?:\d+\.?\d*\s+)?[A-Z][A-Z\s]+|\Z)'
    sections = []
    for match in re.finditer(pattern, json_content, re.MULTILINE | re.DOTALL):
        heading = match.group(1).strip()
        text = match.group(2).strip()
        num_part = re.match(r'^(\d+\.?\d*)', heading)
        level = num_part.group(1).count('.') + 1 if num_part else 0
        sections.append((level, heading, text))
    return sections

def extract_grobid_headings(file_path):
    """Extract headings and their subheadings from GROBID TEI XML file"""
    try:
        # First try parsing as regular file
        try:
            with open(file_path, 'rb') as f:
                tree = etree.parse(f)
        except etree.XMLSyntaxError:
            # If that fails, try gzipped XML
            with gzip.open(file_path, 'rb') as f:
                tree = etree.parse(f)

        ns = {'tei': 'http://www.tei-c.org/ns/1.0'}
        headings = []

        # Find all head elements with @n attribute
        for head in tree.xpath('//tei:body//tei:head[@n]', namespaces=ns):
            n_value = head.get('n')
            head_text = head.xpath('./text()', namespaces=ns)[0].strip()

            # Split the n_value to determine level
            parts = n_value.split('.')
            level = len(parts)

            # If it's a main heading (no decimal)
            if level == 1:
                headings.append({
                    'level': level,
                    'n_value': n_value,
                    'text': head_text,
                    'subheadings': []
                })
            # If it's a subheading (has decimal)
            else:
                # Find the parent heading (n_value without last part)
                parent_n = '.'.join(parts[:-1])
                for heading in headings:
                    if heading['n_value'] == parent_n:
                        heading['subheadings'].append({
                            'level': level,
                            'n_value': n_value,
                            'text': head_text
                        })
                        break

        return headings
    except Exception as e:
        print(f"Error processing GROBID file: {str(e)}")
        return []

def process_files_to_csv(vila_dir, grobid_dir, output_csv, min_score=0.6):
    """Process all file pairs with exact filename matching"""
    vila_dir = Path(vila_dir)
    grobid_dir = Path(grobid_dir)

    # Get all files
    vila_files = list(vila_dir.glob("*.json.gz"))
    grobid_files = list(grobid_dir.glob("*.tei*"))

    # Create exact filename mapping
    grobid_map = {}
    for g_path in grobid_files:
        base = g_path.name.split('.grobid')[0]
        grobid_map[base] = g_path

    # Prepare DataFrame with desired columns
    df = pd.DataFrame(columns=[
        "paper_id",
        "section_name",
        "section_content",
        "subheadings"
    ])

    for v_path in vila_files:
        paper_id = v_path.stem.split('.')[0]  # Get paper ID (e.g., ztMLindFLWR)
        print(f"\nProcessing paper: {paper_id}")

        if paper_id not in grobid_map:
            print(f"No matching GROBID file found for {paper_id}")
            continue

        g_path = grobid_map[paper_id]

        try:
            # Process VILA file
            with gzip.open(v_path, 'rt', encoding='utf-8') as f:
                vila_content = json.load(f)
                symbols_content = vila_content.get("symbols", "")
                vila_sections = extract_vila_sections(symbols_content)

            # Process GROBID file
            grobid_headings = extract_grobid_headings(g_path)

            # For each main heading, find matching VILA section
            for heading in grobid_headings:
                main_heading_text = heading['text']

                # Find best matching VILA section
                best_match = None
                best_score = 0

                for v_level, v_head, v_text in vila_sections:
                    score = SequenceMatcher(
                        None,
                        normalize_for_matching(v_head),
                        normalize_for_matching(main_heading_text)
                    ).ratio()

                    if score > best_score and score >= min_score:
                        best_score = score
                        best_match = (v_head, v_text)

                # Prepare subheadings as semicolon-separated string
                subheadings = "; ".join(
                    f"{sub['n_value']}: {sub['text']}"
                    for sub in heading['subheadings']
                )

                # Add to DataFrame if we found a match
                if best_match:
                    df = pd.concat([df, pd.DataFrame([{
                        "paper_id": paper_id,
                        "section_name": main_heading_text,
                        "section_content": best_match[1],
                        "subheadings": subheadings
                    }])], ignore_index=True)

                    # print(f"Matched section: {main_heading_text[:50]}...")

        except Exception as e:
            print(f"Error processing {paper_id}: {str(e)}")
            continue

    if not df.empty:
        df.to_csv(output_csv, index=False)
        print(f"\nSuccess! Saved results to {output_csv}")
        # print(f"Total papers processed: {len(df['paper_id'].unique())}")
        # print(f"Total sections matched: {len(df)}")
    else:
        print("\nNo matching data found")

if __name__ == "__main__":
    # Mount Google Drive if using Colab
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        pass

    VILA_DIR = "/content/drive/MyDrive/vila_test" #add trajectory to vila file here
    GROBID_DIR = "/content/drive/MyDrive/grobid_test" #add trajectory to grobid file here
    OUTPUT_CSV = "matched_results.csv" # change to desired csv name

    process_files_to_csv(VILA_DIR, GROBID_DIR, OUTPUT_CSV, min_score=0.6)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Processing paper: ztMLindFLWR
Matched section: INTRODUCTION...
Matched section: PRELIMINARIES...
Matched section: PROPOSED MODEL...
Matched section: EXPERIMENTS...

Processing paper: _0kaDkv3dVf
Matched section: INTRODUCTION...
Matched section: RELATED WORKS...
Matched section: ANALYSIS ON HW-NAS-BENCH...
Matched section: CONCLUSION...

Success! Saved results to matched_results.csv
Total papers processed: 2
Total sections matched: 8
